# Notebook 01 — TFIM Data Exploration

Explore the 1D Transverse-Field Ising Model dataset:
- Phase diagram (energy vs h)
- Ground-state entanglement entropy across the phase transition
- ZZ correlator profiles in ordered vs disordered phase
- Comparison of quantum observables

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qsae.datasets import tfim_ground_states, tfim_phase_labels
from qsae.observables import compute_all_observables

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# --- Generate uniform TFIM ground states ---
n = 8
h_values = np.linspace(0.05, 3.0, 80)
print(f'Computing {len(h_values)} ground states for L={n}...')
states = tfim_ground_states(n, h_values, J=1.0)
print(f'states shape: {states.shape}')

In [ ]:
# --- Compute all observables ---
obs = compute_all_observables(states, n, h_values=h_values)
print('Observable keys:', list(obs.keys()))

In [ ]:
# --- Plot: entropy, ZZ, order parameter vs h ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ax = axes[0]
ax.plot(h_values, obs['entropy'], 'b-o', ms=3)
ax.axvline(1.0, color='r', ls='--', lw=1, label='h_c = 1')
ax.set_xlabel('h / J')
ax.set_ylabel('S (nats)')
ax.set_title('Half-chain entanglement entropy')
ax.legend()

ax = axes[1]
ax.plot(h_values, obs['mean_nn_zz'], 'g-o', ms=3)
ax.axvline(1.0, color='r', ls='--', lw=1, label='h_c = 1')
ax.set_xlabel('h / J')
ax.set_ylabel(r'$\langle Z_i Z_{i+1} \rangle$')
ax.set_title('Mean nearest-neighbour ZZ correlator')
ax.legend()

ax = axes[2]
ax.plot(h_values, obs['order_param'], 'm-o', ms=3, label='|<Z>|')
ax.plot(h_values, obs['mean_x'],     'c-o', ms=3, label='<X>')
ax.axvline(1.0, color='r', ls='--', lw=1, label='h_c = 1')
ax.set_xlabel('h / J')
ax.set_ylabel('Observable')
ax.set_title('Order parameter and transverse magnetization')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- ZZ correlator matrix at three field values ---
from qsae.observables import zz_correlator_matrix

h_cases = [0.2, 1.0, 2.5]
h_idx   = [np.argmin(np.abs(h_values - h)) for h in h_cases]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, idx, h in zip(axes, h_idx, h_cases):
    M = zz_correlator_matrix(states[idx], n)
    im = ax.imshow(M, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title(f'h = {h}J  ({"ordered" if h < 1 else "critical" if h == 1 else "disordered"})')
    plt.colorbar(im, ax=ax)

plt.suptitle(r'$\langle Z_i Z_j \rangle$ matrix', y=1.02)
plt.tight_layout()
plt.show()